## Heart Disease Prediction

**Goal:** Predict whether a patient has heart disease based on clinical features.

**Dataset:** UCI Heart Disease — 920 patients, 16 features, numerical target (We will change it to binary).

**Approach:** EDA → Preprocessing Pipeline → 3 Models → Cross-validation → GridSearchCV tuning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('heart_disease_uci.csv')
print(df.shape)
print(df.head())
print(df.isnull().sum())
print(df.dtypes)
categorical_features = ['sex', 'cp', 'restecg', 'fbs', 'exang', 'slope', 'thal']

In [ ]:
## Label Encoding
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])

# Target 'num': 0 = no disease, 1/2/3/4 = disease present
# Binarize to 0 vs 1 for clean binary classification
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['id', 'dataset', 'num'], errors='ignore')

print('Target distribution:')
print(df['target'].value_counts())
print(f'\nClass balance: {df["target"].mean():.2%} positive (heart disease)')

print('\nNull counts:')
print(df.isnull().sum().sort_values(ascending=False))

### Observations
- Target is reasonably balanced (~55% positive) — accuracy is a valid metric here
- Several columns have missing values: `thal`, `ca`, `slope`, `thalch`, `oldpeak`, `chol`, `trestbps`, `fbs`, `restecg`, `exang`
- Mixed types: numeric (`age`, `trestbps`, `chol`) and categorical (`cp`, `restecg`, `thal`)
- Pipeline will handle all nulls — no manual filling needed

## Step 1: EDA
- Visualize Numeric Features
- Visualize Target
- Visualize Categorical Features

In [ ]:
numeric_features = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'sex']

fig, axes = plt.subplots(4, 2, figsize = (10, 10))
axes = axes.flatten()
for i, feature in enumerate(numeric_features):
    axes[i].hist(df[feature], bins = 30, color = 'Steelblue', edgecolor = 'black', linewidth = 0.5)
    axes[i].set_title(feature, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
axes[-1].set_axis_off()
plt.suptitle('Distribution of Numeric Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

#### Display Target Values
fig, ax = plt.subplots(figsize = (8 , 6))
counts = df.groupby('target').target.count()
ax.bar(['No Disease (0)', 'Heart Disease (1)'], counts.values, color=['steelblue', 'coral'], edgecolor='black', width=0.5)
ax.set_title('Heart Disease Balance', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

categorical_features = ['fbs', 'exang', 'cp', 'restecg', 'slope', 'thal']

fig, axes = plt.subplots(3, 2, figsize = (10, 10))
axes = axes.flatten()
for i, feature in enumerate(categorical_features):
    counts = df[feature].value_counts()
    axes[i].bar(counts.index, counts.values, color = 'Steelblue', edgecolor = 'black', width = 0.6)
    axes[i].set_title(feature, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Category')
    axes[i].set_ylabel('Count')
plt.suptitle('Distribution of Categorical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 2: Preprocessing Pipeline
- Split data into train and test before starting with pielines

**Preprocessing Decisions made:**
- **Numeric features:** impute with median (robust to outliers), then StandardScaler
- **Categorical features:** impute with most_frequent, then OneHotEncoder.

### Split data in Train and Test

In [ ]:
X = df.drop(columns = ['target'], errors='ignore')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train Size: {X_train.shape[0]}')
print(f'Test Size: {X_test.shape[0]}')

### Preprocessing Pieline

In [ ]:
numeric_pipeline = Pipeline([
    ('numeric_imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, drop='first'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

## Step 3: Train 3 models
- Logistic Regression
- Decision Tree
- Random Forest

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter = 1000))
])

dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

## Step 4: Cross Validation & Best Model Evaluation

### Compare the Cross Validation Scores of each Model.
- CV scores each fold
- Mean CV score
- Std of Cv score

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv_scores = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'CV Scores of Linear Regression each fold: {np.round(lr_cv_scores, 4)}')
print(f'Mean CV Scores for Linear Regression: {np.round(lr_cv_scores.mean(), 4)}')
print(f'Std CV Scores of Linear Regression: {np.round(lr_cv_scores.std(), 4)}')

dt_cv_scores = cross_val_score(dt_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'CV Scores of Decision Tree each fold: {np.round(dt_cv_scores, 4)}')
print(f'Mean CV Scores for Decision Tree: {np.round(dt_cv_scores.mean(), 4)}')
print(f'Std CV Scores of Decision Tree: {np.round(dt_cv_scores.std(), 4)}')

rf_cv_scores = cross_val_score(rf_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'CV Scores of Random Forest each fold: {np.round(rf_cv_scores, 4)}')
print(f'Mean CV Scores for Random Forest: {np.round(rf_cv_scores.mean(), 4)}')
print(f'Std CV Scores of Random Forest: {np.round(rf_cv_scores.std(), 4)}')

### Evaluate the Best Model
- Accuracy Score
- Classification Report
- Confusion Matrix Display

In [ ]:
## Print Classification report
best_pipeline = rf_pipeline
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
print(f'Accuracy Score: {accuracy_score(y_test, y_pred):.2f}')
print(f'Classification Report: {classification_report(y_test, y_pred, target_names=['No Disease', 'Heart Disease'])}')
fig, ax = plt.subplots(figsize = (8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Disease', 'Heart Disease'], cmap='Blues', ax = ax)
ax.set_title('Confusion Matrix for Best Model (Random Forest)', fontsize = 15, fontweight = 'bold')
plt.tight_layout()
plt.show()

## Step 5:  Tune the best model with GridSearchCV
Tuning Random Forest — the strongest model from CV comparison.

In [ ]:
param_grid = {
    'model__max_depth': [4, 6, 8, None],
    'model__n_estimators': [50, 100, 200],
    'model__min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(best_pipeline, param_grid, scoring='accuracy', cv=5, verbose=1, n_jobs=-1)

grid_search.fit(X_train, y_train)
print(f'Best Hyperparameters for Random Forest: {grid_search.best_params_}')
print(f'Best CV score for Random Forest: {grid_search.best_score_:.4f}')
print(f'Test set score:  {grid_search.best_estimator_.score(X_test, y_test):.4f}')

## Summary & Conclusions

| Model               | CV Accuracy | Notes                              |
|---------------------|-------------|------------------------------------|
| Logistic Regression | ~0.82       | Strong linear baseline             |
| Decision Tree       | ~0.73       | Interpretable but overfits         |
| Random Forest       | ~0.82       | Best performance, most robust      |

**Best model:** Random Forest with lower std
